Looking at the files generated

### Top level view of the generated files and folders: Counts

In [1]:
import json
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 60)
DATA = Path("data")

rows = []
for folder in sorted(p for p in DATA.rglob("*") if p.is_dir()):
    files = [f for f in folder.rglob("*") if f.is_file()]
    rows.append({"folder": str(folder.relative_to(DATA)), "files": len(files),
                 "MB": round(sum(f.stat().st_size for f in files) / 1e6, 1)})
pd.DataFrame(rows).query("folder.str.count('/') < 2")

,folder,files,MB
0,biotools,3,0.1
1,download,3282,792.2
2,download/crates,787,736.6
3,download/ga,876,37.1
791,download/meta,1617,18.3
792,galaxy,3,0.6
793,mapping,2,3.9


The 1617 files in download/meta tells us that we sucessfully downloaded 1617/1620 workflows from workflowhub. I got a couple of errors while running so that makes sense. 
The 787 files in download/crates represent all uploaded workflows that contain ga files (some of which have multple) and download/ga have the final number of ga workflows extracted from these crates: 876. 


In [2]:
index = json.loads((DATA / "download/index.json").read_text())
wf = pd.read_csv(DATA / "download/workflows.csv", dtype={"id": str})
print("rows in workflows.csv:", len(wf))

rows in workflows.csv: 1617


The index contains notes of all 1620 workflows on workflowhub. IDs are automatically generated - they start at 2 and go to 2374, so they might not look like IDs but they are.
Workflows.csv contains the 1617 successful downloads, which matches download/meta - makes sense

In [3]:
wf["workflow_class"].value_counts()

workflow_class
Galaxy                                      788
Nextflow                                    212
Common Workflow Language                    123
Snakemake                                    83
COMP Superscalar                             75
Python                                       66
Unrecognized workflow type                   47
Jupyter                                      46
Workflow Description Language                33
Docker                                       27
Scipion                                      20
Shell Script                                 19
KNIME                                        15
Apache Airflow                               15
R markdown                                   10
Autosubmit                                   10
Playbook Workflow Builder Workflow            9
Bpipe                                         5
Quarto Markdown                               2
BioCompute Object                             2
Argo Workflow            

It says 788 galaxy workflows, so probably 1 of the errors we see in the 1617/1620 was a galaxy workflow. 

### Checking which ones have multiple files and which ones have none


In [4]:
galaxy = wf[wf["workflow_class"].str.lower() == "galaxy"].copy()
galaxy["has_ga"] = galaxy["ga_files"].notna()
galaxy["n_ga"] = galaxy["ga_files"].fillna("").str.split(";").map(lambda x: len([g for g in x if g]))
print(galaxy["has_ga"].value_counts())
print("\nGalaxy workflows with more than one .ga:", (galaxy["n_ga"] > 1).sum())
galaxy.loc[~galaxy["has_ga"], ["id", "title"]]

has_ga
True     779
False      9
Name: count, dtype: int64

Galaxy workflows with more than one .ga: 31


,id,title
458,685,mRNA-Seq BY-COVID Pipeline
697,1113,Full Analyse Argo data
805,1314,Galaxy Workflow for CNV Detection with CNVkit and Conver...
1396,2083,Example deep metabolome annotation LC-MS/MS data process...
1397,2084,Deep metabolome annotation LC-MS/MS fractionation and DI...
1398,2085,Deep metabolome annotation LC-MS/MS workflow (W2)
1399,2086,Deep metabolome annotation LC-MS/MS (metabolite standard...
1464,2167,GlyCombo Galaxy
1596,2350,AOPTK download and analysis


Numbers add up! TODO: perhaps we should look into/exclude the ones that have multiples - could be same workflow repeated, for example the ones wth multiple verisons

In [ ]:
# Looking at most common topics for galaxy workflows - we are only interested in life science ones
galaxy["topics"].dropna().str.split(";").explode().value_counts().head(20)

topics
Whole genome sequencing    15
Sequence assembly          14
Ecology                    12
Biodiversity               10
Genetic variation           8
Transcriptomics             8
Biomedical science          8
Synthetic biology           8
Population genomics         7
Gene expression             6
Bioinformatics              6
Genomics                    6
Metagenomics                5
Virology                    5
Bioimaging                  4
Imaging                     4
Microbiology                4
Physics                     4
Metabolomics                4
Oncology                    3
Name: count, dtype: int64

### Looking at edges extracted by `galaxy_tool_pairs.py`

In [24]:
pairs = pd.read_csv(DATA / "galaxy/galaxy_pairs.csv", dtype={"workflow_id": str})
gedges = pd.read_csv(DATA / "galaxy/galaxy_edges.csv")
gtools = pd.read_csv(DATA / "galaxy/galaxy_tools.csv")
print("Nr connections:", len(pairs), ", nr unique connections:", len(gedges), f", nr tools:", len(gtools))
print("workflows with at least one pair:", pairs["workflow_id"].nunique(), "of", galaxy["has_ga"].sum())
print("self-loops", (pairs["source"] == pairs["target"]).sum())

Nr connections: 5091 , nr unique connections: 2182 , nr tools: 1186
workflows with at least one pair: 629 of 779
self-loops 0


So 150 workflows out of 779 did not have any identifiable edges. Should check which of these were, it might just be that they simply had input -> tool -> output, in which case the code would not collect any edges.
Self loops == 0 is as it should be! 

#### Looking at the ones with no edges

In [51]:
# workflows with a .ga but no pairs: one tool only, or something missed?
no_pairs = galaxy[galaxy["has_ga"] & ~galaxy["id"].isin(pairs["workflow_id"])]
print(len(no_pairs))

# printing the links to each so I cancheck them out
ids = no_pairs["id"].head(10).to_dict().values()
for wf_id in ids:
    print(f"https://workflowhub.eu/workflows/{wf_id}?version=1")

no_pairs[["id", "title"]].head(10)

150
https://workflowhub.eu/workflows/16?version=1
https://workflowhub.eu/workflows/22?version=1
https://workflowhub.eu/workflows/23?version=1
https://workflowhub.eu/workflows/24?version=1
https://workflowhub.eu/workflows/25?version=1
https://workflowhub.eu/workflows/123?version=1
https://workflowhub.eu/workflows/137?version=1
https://workflowhub.eu/workflows/189?version=1
https://workflowhub.eu/workflows/193?version=1
https://workflowhub.eu/workflows/194?version=1


,id,title
14,16,Cheminformatics - TransFS scoring
19,22,Pathway Analysis
20,23,Genetic Design
21,24,RetroSynthesis
22,25,Pathway Ranker
92,123,Compute daily and monthly mean from meteorological stati...
103,137,parallel-accession-download/main
125,189,BioTranslator Workflow
127,193,Workflow of BioTranslator Comparative Analysis
128,194,Protein MD Setup tutorial using BioExcel Building Blocks...


Comments per workflow
```
14    16                            Cheminformatics - TransFS scoring       has a bunch of seemingly non tool nodes
19    22                                             Pathway Analysis       no preview  
20    23                                               Genetic Design       strange - seems to contain things that are in the maping document, also no preview 
21    24                                               RetroSynthesis       many steps no recognised tools, but that should not matter
22    25                                               Pathway Ranker       -"-
92   123  Compute daily and monthly mean from meteorological stati...       -"-
103  137                             parallel-accession-download/main       -"-
125  189                                       BioTranslator Workflow       -"-
127  193               Workflow of BioTranslator Comparative Analysis       only one tool
128  194  Protein MD Setup tutorial using BioExcel Building Blocks...       many steps no recognised tools, but that should not matter
```

I dont know what the conclusion was - maybe need to look into how the files look more. Currently only looked at the overview with the step names and visualisations. I dont feel like doing that right now though soo TODO



In [8]:
pairs.groupby("workflow_id").size().describe()

count    629.000000
mean       8.093800
std        9.898487
min        1.000000
25%        2.000000
50%        5.000000
75%       11.000000
max       98.000000
dtype: float64

Of those 629 workflows that do have edges extracted, the mean nr of edges is ~ 8, with a standatrd deviation of ~ 10 edges and the max nr edges in a workflow is 98.

In [ ]:
# Most common edges
gedges.head(15)

,source,target,n_workflows,workflow_ids
0,fastqc,multiqc,41,2;4;222;226;230;346;472;482;601;602;628;676;688;876;1061...
1,fastp,multiqc,37,2;4;7;8;35;36;37;38;39;40;110;112;113;155;346;353;357;39...
2,fastp,bwa,22,2;4;7;35;36;38;39;40;110;113;155;353;357;439;517;518;119...
3,samtools_stats,multiqc,18,7;110;111;113;155;347;351;353;357;439;519;625;1190;1491;...
4,picard,multiqc,16,7;8;112;113;353;357;395;399;1190;1540;1642;1692;1695;171...
5,trimmomatic,fastqc,16,232;233;628;676;876;1271;1272;1274;1651;1678;1685;1716;1...
6,bwa,samtools_view,15,110;113;155;353;357;439;628;676;1190;1260;1540;1651;1668...
7,meryl,genomescope,15,223;226;230;309;365;366;603;631;698;1103;1104;1607;1608;...
8,scanpy_run_umap,scanpy_find_cluster,13,465;467;468;512;513;514;646;647;1356;1362;1490;1496;1510
9,flye,bandage,12,225;226;230;750;788;1062;1114;1495;1595;1611;1613;1617


### Looking into the different translation methods

In [56]:
# translation map
tmap = pd.read_csv(DATA / "mapping/galaxy_tool_map.csv")
tmap["method"] = tmap["method"].fillna("unmapped")
mapped = tmap["method"] != "unmapped"
print("total nr unique tools:", len(tmap), ", nr tools mapped:", mapped.sum(), ", nr tools not mapped:", (~mapped).sum())
print("total nr uses of any tool:", tmap["n_workflows"].sum(), " of which are mapped:", tmap.loc[mapped, "n_workflows"].sum(),
      " and not mapped:", tmap.loc[~mapped, "n_workflows"].sum())
summary = tmap.groupby("method").agg(tools=("tool", "size"), tool_uses=("n_workflows", "sum"))
summary["share_of_tools"] = (summary["tools"] / summary["tools"].sum()).round(2)
summary["share_of_uses"] = (summary["tool_uses"] / summary["tool_uses"].sum()).round(2)
summary.sort_values("tool_uses", ascending=False)

total nr unique tools: 1186 , nr tools mapped: 851 , nr tools not mapped: 335
total nr uses of any tool: 4419  of which are mapped: 3408  and not mapped: 1011


,tools,tool_uses,share_of_tools,share_of_uses
method,,,,
codex,515,2307,0.43,0.52
unmapped,335,1011,0.28,0.23
prefix,275,869,0.23,0.20
name,61,232,0.05,0.05


Out of the total 1186 (TODO: check - wasnt there like 2000+?) tools, ~50% were mapped by galaxy codex (the map not the ai) ~ 25% was unmapped, ~20 % was mapped using prefix and the last 5 % by name (i guess the last one is becuse the codex mappings took most clear name matches)

#### It is possible we could use some paper dois to match to bio.tools

In [11]:
print("mapped tools with a paper:", tmap.loc[tmap["method"] != "unmapped", "has_publication"].mean().round(2))
tmap[tmap["disagree"]][["tool", "n_workflows", "codex_id", "name_id"]]

mapped tools with a paper: 0.83


,tool,n_workflows,codex_id,name_id
28,taxonomy_krona_chart,20,krona,taxonomy_krona_chart
106,eggnog_mapper,8,eggnog-mapper-v2,eggnog-mapper
170,circos,5,galactic_circos,circos
172,clustalw,5,clustal2,clustalw
267,metaspades,4,spades,metaspades
397,msstats,3,msstatstmt,msstats
463,alevin,2,salmon,Alevin
488,checkm2,2,checkm,checkm2
497,control_freec,2,freec,control-freec
654,pangolin,2,pangolin_cov-lineages,pangolin


~85 % so yeah we should try this. TODO

#### Most commonly used tools identified via prefix method

In [12]:
tmap[tmap["method"] == "prefix"].sort_values("n_workflows", ascending=False)[["tool", "n_workflows", "biotoolsID"]].head(20)

,tool,n_workflows,biotoolsID
10,samtools_view,32,samtools
17,samtools_stats,24,samtools
34,samtools_fastx,18,samtools
35,anndata_ops,17,anndata
44,openbabel_compound_convert,15,openbabel
51,scanpy_compute_graph,14,scanpy
54,scanpy_plot_embed,14,scanpy
58,scanpy_find_cluster,13,scanpy
59,scanpy_run_umap,13,scanpy
67,scanpy_normalise_data,12,scanpy


Im not able to judge what is a tool or not by just looking at this. TODO: ask magnus. RESULT: magnus says yes.

In [13]:
tmap[tmap["method"] == "unmapped"].sort_values("n_workflows", ascending=False)[["tool", "owner", "n_workflows"]].head(20)

,tool,owner,n_workflows
13,collection_column_join,iuc,29
22,tabular_to_fasta,devteam,22
20,filter_tabular,iuc,22
23,datamash_transpose,iuc,21
24,fasta_merge_files_and_filter_unique_sequences,galaxyp,21
25,fasta_stats,iuc,21
38,table_compute,iuc,17
39,fasta_to_tabular,devteam,16
40,genomescope,iuc,16
47,fasta_formatter,devteam,14


Im not able to judge what is a tool or not by just looking at this. TODO: ask magnus. 

In [14]:
# bio.tools IDs that many different ToolShed tools map to (suites, or a wrong match swallowing tools)
tmap[tmap["method"] != "unmapped"].groupby("biotoolsID")["tool"].agg(["size", list]).sort_values("size", ascending=False).head(15)

,size,list
biotoolsID,,
mothur,34,"[mothur_summary_seqs, mothur_screen_seqs, mothur_unique_..."
openms,29,"[openms_falsediscoveryrate, openms_peptideindexer, openm..."
scanpy,27,"[scanpy_plot, scanpy_compute_graph, scanpy_filter, scanp..."
CellProfiler,23,"[cp_cellprofiler, cp_common, cp_export_to_spreadsheet, c..."
qiime2,23,"[qiime2_core__tools__import, qiime2__feature_classifier_..."
galaxy_image_analysis,20,"[2d_auto_threshold, binary2labelimage, 2d_simple_filter,..."
seurat,17,"[seurat_clustering, seurat_data, seurat_plot, seurat_pre..."
samtools,12,"[samtools_view, samtools_stats, samtools_fastx, samtools..."
hicexplorer,10,"[hicexplorer_hicmergematrixbins, hicexplorer_hicplotmatr..."


### Looking at the translated files

In [ ]:
# lookign at how many workflows contain exactly the same edges in the same order
bio_pairs = pd.read_csv(DATA / "biotools/biotools_pairs.csv", dtype={"workflow_id": str})

structures = {}
for wf_id, g in bio_pairs.groupby("workflow_id"):
    fingerprint = tuple(sorted(zip(g["source"], g["target"])))
    structures.setdefault(fingerprint, []).append(wf_id)

shared = {fp: ids for fp, ids in structures.items() if len(ids) > 1}
print("workflows:", sum(len(ids) for ids in structures.values()),
      ", distinct structures:", len(structures),
      ", structures shared by >1 workflow:", len(shared),
      ", extra copies:", sum(len(ids) - 1 for ids in shared.values()))

workflows: 413 , distinct structures: 317 , structures shared by >1 workflow: 72 , extra copies: 96


Not too bad in terms of repetition

In [ ]:
edges = pd.read_csv(DATA / "biotools/biotools_edges.csv")
nodes = pd.read_csv(DATA / "biotools/biotools_nodes.csv")
print("translated edges:", len(edges), ",  translated nodes:", len(nodes))
print("nodes without a paper (probably wont be able to identify as tools unless each is individually evaluated):", (~nodes["has_publication"]).sum())

# nr of times the most used edges are used 
edges["n_workflows"].value_counts().sort_index().head(10)

translated edges: 758 ,  translated nodes: 323
nodes without a paper (probably wont be able to identify as tools unless each is individually evaluated): 78


n_workflows
1     381
2     181
3      72
4      41
5      26
6      18
7       6
8       9
9       8
10      1
Name: count, dtype: int64

Why is it so few edges - didnt we have more? TODO: check RESULT: I think I was just mixing it up with the pairs file

In [ ]:
# confidence of an edge = weaker of its two tools (codex > name > prefix)
rank = {"codex": 0, "name": 1, "prefix": 2}
# pick out only mapped tools, group by Id and assign it the ranking score
method_of = tmap[tmap["method"] != "unmapped"].groupby("biotoolsID")["method"].agg(lambda m: max(m, key=rank.get))
# for each edge, assign it to the least trustworthy method
edges["confidence"] = [max(method_of.get(s), method_of.get(t), key=rank.get) for s, t in zip(edges["source"], edges["target"])]
edges["confidence"].value_counts()

758


confidence
codex     443
prefix    223
name       92
Name: count, dtype: int64

Assigning each edge a confidence score, we have 443 connections that are probable to be true 

In [65]:
# Based on the results above, creating three new edges files - one for each confidence level so we can use those to compare to the graph
levels = {
    "codex": ["codex"],
    "codex_name": ["codex", "name"],
    "all": ["codex", "name", "prefix"],
}

for level, allowed in levels.items():
    subset = edges[edges["confidence"].isin(allowed)]
    out = DATA / f"biotools/biotools_edges_{level}.csv"
    subset.to_csv(out, index=False)
    print(f"{out.name}: {len(subset)} edges")

biotools_edges_codex.csv: 443 edges
biotools_edges_codex_name.csv: 535 edges
biotools_edges_all.csv: 758 edges


In [18]:
# pairs seen in both directions across workflows
rev = edges.merge(edges, left_on=["source", "target"], right_on=["target", "source"], suffixes=("", "_rev"))
print("edges with a reverse edge:", len(rev))
rev[["source", "target", "n_workflows", "n_workflows_rev"]].sort_values("n_workflows", ascending=False).head(10)

edges with a reverse edge: 92


,source,target,n_workflows,n_workflows_rev
0,anndata,scanpy,24,24
1,scanpy,anndata,24,24
2,galaxy_image_analysis,scikit-image,11,11
3,minimap2,Racon,11,9
4,scikit-image,galaxy_image_analysis,11,11
5,Racon,minimap2,9,11
6,lofreq,snpeff,9,6
9,picard_samtofastq,samtools,8,6
10,purge_dups,minimap2,8,8
8,minimap2,purge_dups,8,8


Suspiciously symmetrical numbers but we spot checked (n=1 :D https://workflowhub.eu/workflows/647?version=1) and scanpy and anndata were actually being used back and forth so its probably fine :S 

In [19]:
# hubs: tools with the most incoming / outgoing edges
pd.DataFrame({"out": edges["source"].value_counts(), "in": edges["target"].value_counts()}).fillna(0).astype(int).sort_values("in", ascending=False).head(15)

,out,in
multiqc,0,32
tooldistillator,0,19
busco,2,15
samtools,31,14
fastqc,3,14
jbrowse,1,14
bcftools,6,13
quast,2,12
bedtools,12,11
bwa,14,11
